# TESS SPOC PDC-SAP Availability Query
Queries TESS SPOC 2-minute cadence availability for the top-5 Hot Jupiter and top-5 Ultra-Hot Jupiter planets selected by transit depth from TEPCat.

In [ ]:
import os
import pandas as pd
import lightkurve as lk
from IPython.display import display

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
FOLDER   = os.path.join(BASE_DIR, 'Final HJ and UHJ')

# ── Batch 1 ───────────────────────────────────────────────────────────────────
hj_b1  = pd.read_csv(os.path.join(FOLDER, 'hot_jupiters_top5.csv'))
uhj_b1 = pd.read_csv(os.path.join(FOLDER, 'ultra_hot_jupiters_top5.csv'))

# ── Batch 2 ───────────────────────────────────────────────────────────────────
hj_b2  = pd.read_csv(os.path.join(FOLDER, 'hot_jupiters_batch2.csv'))
uhj_b2 = pd.read_csv(os.path.join(FOLDER, 'ultra_hot_jupiters_batch2.csv'))

# Add batch label
for df, label in [(hj_b1,'HJ-B1'),(uhj_b1,'UHJ-B1'),(hj_b2,'HJ-B2'),(uhj_b2,'UHJ-B2')]:
    df['batch'] = label

hj_all  = pd.concat([hj_b1,  hj_b2],  ignore_index=True)
uhj_all = pd.concat([uhj_b1, uhj_b2], ignore_index=True)

cols = ['Planet', 'Transit_depth_pct', 'Period_days', 'Radius_Rjup', 'batch']

print('=== Hot Jupiters (all batches) ===')
display(hj_all[cols].to_string(index=False))
print()
print('=== Ultra-Hot Jupiters (all batches) ===')
display(uhj_all[cols].to_string(index=False))

## Query function
For each planet, we query MAST via `lightkurve` for TESS SPOC 2-minute cadence (`pdcsap_flux`) light curves.

In [3]:
def query_planet(planet_name):
    """Return a DataFrame of available TESS SPOC 2-min sectors for a planet."""
    try:
        sr = lk.search_lightcurve(
            planet_name,
            mission='TESS',
            cadence=120,
            author='SPOC'
        )
        if len(sr) == 0:
            return None, 'No TESS SPOC 2-min data found'
        
        tbl = sr.table.to_pandas()
        keep_cols = []
        for c in ['target_name', '#obs_id', 'sequence_number', 't_min', 't_max', 'exptime']:
            if c in tbl.columns:
                keep_cols.append(c)
        
        # Try to extract sector numbers
        if 'sequence_number' in tbl.columns:
            tbl = tbl.rename(columns={'sequence_number': 'sector'})
        elif '#obs_id' in tbl.columns:
            tbl['sector'] = tbl['#obs_id'].str.extract(r's(\d+)', expand=False)
        
        # Round time columns for readability
        for col in ['t_min', 't_max']:
            if col in tbl.columns:
                tbl[col] = tbl[col].round(2)
        
        out_cols = [c for c in ['target_name', 'sector', 't_min', 't_max', 'exptime'] if c in tbl.columns]
        return tbl[out_cols], f'{len(tbl)} sector(s) found'
    
    except Exception as e:
        return None, f'ERROR: {e}'

## Hot Jupiters — TESS SPOC 2-min Availability

In [ ]:
hj_summary = []

for _, row in hj_all[hj_all['batch'] == 'HJ-B2'].iterrows():
    planet = row['Planet']
    depth  = row['Transit_depth_pct']
    batch  = row['batch']
    print(f'Querying: {planet}  [{batch}]  (depth={depth:.3f}%)')

    tbl, msg = query_planet(planet)
    print(f'  -> {msg}')

    if tbl is not None:
        display(tbl)
        sectors = sorted(tbl['sector'].dropna().astype(str).tolist()) if 'sector' in tbl.columns else []
        hj_summary.append({
            'Planet': planet, 'Type': 'HJ', 'Batch': batch,
            'Transit_depth_pct': depth, 'N_sectors': len(tbl),
            'Sectors': ', '.join(sectors), 'Status': 'Available'
        })
    else:
        hj_summary.append({
            'Planet': planet, 'Type': 'HJ', 'Batch': batch,
            'Transit_depth_pct': depth, 'N_sectors': 0,
            'Sectors': '', 'Status': msg
        })
    print()

## Ultra-Hot Jupiters — TESS SPOC 2-min Availability

In [ ]:
uhj_summary = []

for _, row in uhj_all[uhj_all['batch'] == 'UHJ-B2'].iterrows():
    planet = row['Planet']
    depth  = row['Transit_depth_pct']
    batch  = row['batch']
    print(f'Querying: {planet}  [{batch}]  (depth={depth:.3f}%)')

    tbl, msg = query_planet(planet)
    print(f'  -> {msg}')

    if tbl is not None:
        display(tbl)
        sectors = sorted(tbl['sector'].dropna().astype(str).tolist()) if 'sector' in tbl.columns else []
        uhj_summary.append({
            'Planet': planet, 'Type': 'UHJ', 'Batch': batch,
            'Transit_depth_pct': depth, 'N_sectors': len(tbl),
            'Sectors': ', '.join(sectors), 'Status': 'Available'
        })
    else:
        uhj_summary.append({
            'Planet': planet, 'Type': 'UHJ', 'Batch': batch,
            'Transit_depth_pct': depth, 'N_sectors': 0,
            'Sectors': '', 'Status': msg
        })
    print()

## Summary Table

In [ ]:
# Load existing batch 1 results and tag with batch label if missing
b1_df = pd.read_csv(os.path.join(FOLDER, 'tess_availability.csv'))
if 'Batch' not in b1_df.columns:
    b1_df['Batch'] = b1_df['Type'].map({'HJ': 'HJ-B1', 'UHJ': 'UHJ-B1'})

# New batch 2 results
b2_df = pd.DataFrame(hj_summary + uhj_summary)

summary_df = pd.concat([b1_df, b2_df], ignore_index=True)
summary_df = summary_df.sort_values(['Type', 'Batch', 'Transit_depth_pct'],
                                     ascending=[True, True, False])

print('=== TESS SPOC 2-min Availability — All Batches ===')
display(summary_df)

out_path = os.path.join(FOLDER, 'tess_availability_all_batches.csv')
summary_df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')